# M&A Due Diligence Walkthrough

This notebook mirrors the narrative in the companion blog post. Each
code cell imports from the shared `mna` package — the CLI at
`cli/invoke.py` calls the same functions, so what you see here matches
what `python -m mna invoke ...` produces.

Run the cells top-to-bottom after `deploy.sh` / `deploy.ps1` completes
and `data/generate.py --seed-all` has finished.

> **Cost note:** This sample deploys billable AWS resources, with Amazon Aurora Serverless v2 as the primary cost driver. Estimated cost for a full walkthrough is under $5 USD. To remove all resources and stop charges, run the cleanup script (`./cleanup.sh` on macOS/Linux, `.\cleanup.ps1` on Windows) when finished.


## Prerequisites

Before running this notebook, complete the following:

1. **Deploy the sample** — run `./deploy.sh` (macOS/Linux) or `.\deploy.ps1` (Windows) from the repository root. The script provisions all AWS resources and seeds synthetic data.
2. **Verify deployment succeeded** — the deploy script runs a post-deploy smoke test automatically. If it passed, you are ready.
3. **Confirm environment** — the next cell (Environment validation) checks that all `/mna/*` SSM parameters are populated.


## 1. Environment validation

Verifies the AWS region, credential chain, and every `/mna/*` SSM
parameter published by the CDK stacks. Stop here and re-run deploy if
any parameter is missing.


In [ ]:
import os

import boto3

from mna.config import ALL_PARAMETERS, load_config

session = boto3.Session()
region = session.region_name or os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")
credentials = session.get_credentials()

print(f"Region: {region}")
print(f"Credentials resolved: {credentials is not None}")

config = load_config(region_name=region)
for param_name in ALL_PARAMETERS:
    value = config.as_dict()[param_name]
    print(f"  {param_name}: {'OK' if value else 'MISSING'}")


## 2. Data overview

Counts the rows seeded into Aurora by `data/generate.py companies` and
lists the document keys uploaded to the documents S3 bucket. Gives a
quick confidence check that the dataset is present before any agent is
invoked.


In [ ]:
import boto3

from mna.config import load_config

config = load_config()
region = boto3.Session().region_name

rds_data = boto3.client("rds-data", region_name=region)
secrets = boto3.client("secretsmanager", region_name=region)

# Aurora row count via the RDS Data API (no driver needed in the notebook).
aurora_secret_arn = secrets.list_secrets(
    Filters=[{"Key": "name", "Values": ["mna"]}],
).get("SecretList", [{}])[0].get("ARN")

row_count = None
if aurora_secret_arn:
    result = rds_data.execute_statement(
        resourceArn=config.aurora_cluster_arn,
        secretArn=aurora_secret_arn,
        database="postgres",
        sql="SELECT COUNT(*) FROM mna.target_companies",
    )
    row_count = result["records"][0][0]["longValue"]
print(f"Aurora target_companies rows: {row_count}")

# KB document inventory from the documents bucket.
s3 = boto3.client("s3", region_name=region)
paginator = s3.get_paginator("list_objects_v2")
doc_keys: list[str] = []
for page in paginator.paginate(Bucket=config.docs_bucket):
    doc_keys.extend(obj["Key"] for obj in page.get("Contents", []))
print(f"KB document objects in s3://{config.docs_bucket}: {len(doc_keys)}")
for key in doc_keys[:10]:
    print(f"  {key}")


## 3. Target Screening — text-to-SQL + KB enrichment


In [ ]:
from IPython.display import Markdown, display

from mna.client import invoke_agent

PROMPT = (
    "Screen our target pipeline for transportation companies with revenue between $100M and $500M, EBITDA margin above 12%, and fleet size above 200. Surface the top three and tell me what the CIM says about the leader's growth trajectory."
)

response = invoke_agent("target_screening", PROMPT, session_id="walkthrough")

display(Markdown(f"**Response:**\n\n{response.text}"))

if response.citations:
    display(Markdown("**Citations:**"))
    for idx, cite in enumerate(response.citations, start=1):
        location = cite.source
        if cite.page is not None:
            location += f", p. {cite.page}"
        display(Markdown(f"{idx}. `{location}` — {cite.text[:160]}"))
else:
    display(Markdown("_No citations returned._"))

print(f"session_id={response.session_id} trace_id={response.trace_id}")


## 4. Financial Analysis — KB + Gateway-backed tool


In [ ]:
from IPython.display import Markdown, display

from mna.client import invoke_agent

PROMPT = (
    'Run a DCF on Example Corp using the CIM in the knowledge base. Flag any management projection that diverges from historical performance by more than 20%, and pull comparable multiples for transportation-logistics mid-market.'
)

response = invoke_agent("financial_analysis", PROMPT, session_id="walkthrough")

display(Markdown(f"**Response:**\n\n{response.text}"))

if response.citations:
    display(Markdown("**Citations:**"))
    for idx, cite in enumerate(response.citations, start=1):
        location = cite.source
        if cite.page is not None:
            location += f", p. {cite.page}"
        display(Markdown(f"{idx}. `{location}` — {cite.text[:160]}"))
else:
    display(Markdown("_No citations returned._"))

print(f"session_id={response.session_id} trace_id={response.trace_id}")


## 5. Strategic Fit — long-term memory over prior deals


In [ ]:
from IPython.display import Markdown, display

from mna.client import invoke_agent

PROMPT = (
    "Compare Example Corp' integration profile against our three most recent completed acquisitions. Identify the top three integration risks and cite the source memos."
)

response = invoke_agent("strategic_fit", PROMPT, session_id="walkthrough")

display(Markdown(f"**Response:**\n\n{response.text}"))

if response.citations:
    display(Markdown("**Citations:**"))
    for idx, cite in enumerate(response.citations, start=1):
        location = cite.source
        if cite.page is not None:
            location += f", p. {cite.page}"
        display(Markdown(f"{idx}. `{location}` — {cite.text[:160]}"))
else:
    display(Markdown("_No citations returned._"))

print(f"session_id={response.session_id} trace_id={response.trace_id}")


## 6. Compliance Validation — evaluator invocation


In [ ]:
from IPython.display import Markdown, display

from mna.client import invoke_agent

PROMPT = (
    'Review the Example Corp analysis in this session for completeness against our M&A governance checklist. List any claims without source citations.'
)

response = invoke_agent("compliance_validation", PROMPT, session_id="walkthrough")

display(Markdown(f"**Response:**\n\n{response.text}"))

if response.citations:
    display(Markdown("**Citations:**"))
    for idx, cite in enumerate(response.citations, start=1):
        location = cite.source
        if cite.page is not None:
            location += f", p. {cite.page}"
        display(Markdown(f"{idx}. `{location}` — {cite.text[:160]}"))
else:
    display(Markdown("_No citations returned._"))

print(f"session_id={response.session_id} trace_id={response.trace_id}")


## 7. Trace inspection

Fetches the X-Ray trace for the most recent invocation so you can see
the supervisor → specialist → tool hierarchy described in the blog.
Re-run the cell with a different `trace_id` to inspect earlier turns.


In [ ]:
from mna.client import get_last_trace

# Replace with the trace_id printed by the cell above whose trace you
# want to inspect. Defaults to the Compliance Validation invocation.
trace_id = response.trace_id

if not trace_id:
    print("No trace_id available — run a specialist cell first.")
else:
    trace = get_last_trace(trace_id)
    summary = trace.get("summary") or {}
    segments = trace.get("segments") or []
    print(f"Trace {trace_id}")
    print(f"  duration: {summary.get('Duration')}")
    print(f"  segments: {len(segments)}")
    for seg in segments:
        name = seg.get("name") or "(anonymous)"
        origin = seg.get("origin") or "Unknown"
        print(f"    - {name} ({origin})")


## 8. Cleanup

When you finish exploring the sample, remove all deployed resources to stop charges:

```bash
# macOS / Linux:
./cleanup.sh

# Windows:
.\cleanup.ps1
```

The cleanup script will:

1. Prompt for confirmation before any destructive action.
2. Run `cdk destroy --all --force` to remove every stack.
3. Verify that billable resources were removed.
4. Print commands you can run to double-check cleanup succeeded.

Estimated cost for a full walkthrough (deploy, run prompts, cleanup) is under $5 USD, with Amazon Aurora Serverless v2 as the primary cost driver.


## 9. Conclusion

You have now exercised the full multi-agent M&A due-diligence workflow:

- **Target Screening** — text-to-SQL on Amazon Aurora plus narrative enrichment from the knowledge base.
- **Financial Analysis** — DCF grounded in KB passages with market-data comparables via the AgentCore Gateway tool.
- **Strategic Fit** — long-term memory retrieval from prior-deal memos.
- **Compliance Validation** — governance-checklist audit with the citation-check evaluator.

Each invocation produced an X-Ray trace you can inspect for the full call hierarchy. Adapt the prompts, add new specialists, or extend the tool set by following `CONTRIBUTING.md`.
